In [1]:
import pandas as pd
import numpy as np

# Load the cleaned TED Talks dataset
df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Display the first five rows
df.head()

,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript,combined_text
0,1,averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,with the same humor and humanity he exuded in ...,"thank you so much, chris. and it's truly a gre...",averting the climate crisis ['alternative ener...
1,92,the best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['africa', 'asia', 'google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,you've never seen data presented like this. wi...,"about 10 years ago, i took on the task to teac...","the best stats you've ever seen ['africa', 'as..."
2,7,simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,new york times columnist david pogue takes aim...,"(music: ""the sound of silence,"" simon & garfun...","simplicity sells ['computers', 'entertainment'..."
3,53,greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['macarthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"in an emotionally charged talk, macarthur-winn...",if you're here today — and i'm very happy that...,"greening the ghetto ['macarthur grant', 'activ..."
4,66,do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,sir ken robinson makes an entertaining and pro...,good morning. how are you? (audience) good. it...,"do schools kill creativity? ['children', 'crea..."


In [2]:
# Display all column names
df.columns.tolist()

['talk_id',
 'title',
 'speaker_1',
 'all_speakers',
 'occupations',
 'about_speakers',
 'views',
 'recorded_date',
 'published_date',
 'event',
 'native_lang',
 'available_lang',
 'comments',
 'duration',
 'topics',
 'related_talks',
 'url',
 'description',
 'transcript',
 'combined_text']

In [4]:
df["combined_text"].isnull().sum()

np.int64(0)

"We used this code to identify missing values in the combined text column before applying TF-IDF vectorization."

In [5]:
df["combined_text"] = df["combined_text"].fillna("")

"We replaced missing values in the combined text column with empty strings to prepare the dataset for TF-IDF vectorization.""We replaced missing values in the combined text column with empty strings to prepare the dataset for TF-IDF vectorization."

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

"We imported TfidfVectorizer from Scikit-learn to convert TED Talk text into numerical TF-IDF features."

In [7]:
# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Convert combined text into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df["combined_text"])

# Display the shape of the TF-IDF matrix
tfidf_matrix.shape

(4005, 5000)

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all TED Talks
cosine_sim = cosine_similarity(tfidf_matrix)

# Display the shape
cosine_sim.shape

(4005, 4005)

In [9]:
df[["title"]].head(10)

,title
0,averting the climate crisis
1,the best stats you've ever seen
2,simplicity sells
3,greening the ghetto
4,do schools kill creativity?
5,behind the design of seattle's library
6,letting go of god
7,let's teach religion -- all religion -- in sch...
8,a life of purpose
9,my wish: a global day of film


In [10]:
def recommend_talks(title, top_n=5):

    # Check whether the given title exists
    if title not in df["title"].values:
        print("TED Talk title not found. Please enter a valid title.")
        return pd.DataFrame()

    # Get the index of the selected TED Talk
    talk_index = df[df["title"] == title].index[0]

    # Get similarity scores for the selected talk
    similarity_scores = list(enumerate(cosine_sim[talk_index]))

    # Sort talks by similarity score in descending order
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Exclude the selected talk itself
    similarity_scores = [
        score for score in similarity_scores if score[0] != talk_index
    ]

    # Select the top similar talks
    top_talks = similarity_scores[:top_n]

    # Get recommended talk details
    recommendations = df.iloc[[i[0] for i in top_talks]][
        ["title", "topics"]
    ].copy()

    # Add similarity scores
    recommendations["similarity_score"] = [
        i[1] for i in top_talks
    ]

    return recommendations.reset_index(drop=True)

In [11]:
# Select the first TED Talk title
sample_title = df["title"].iloc[0]

# Generate recommendations
recommendations = recommend_talks(sample_title)

# Display the results
print("Selected TED Talk:", sample_title)
recommendations

Selected TED Talk: averting the climate crisis


,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.368081
1,how we can turn the tide on climate,"['climate change', 'environment', 'activism', ...",0.354537
2,a climate solution where all sides can win,"['big problems', 'business', 'china', 'economi...",0.339493
3,a one-man world summit,"['comedy', 'global issues', 'government', 'per...",0.338872
4,how we could change the planet's climate future,"['climate change', 'sustainability', 'environm...",0.338441


In [12]:
# Select another TED Talk title
second_title = df["title"].iloc[10]

# Generate recommendations
second_recommendations = recommend_talks(second_title)

# Display the results
print("Selected TED Talk:", second_title)
second_recommendations

Selected TED Talk: my wish: help me stop pandemics


,title,topics,similarity_score
0,the case for optimism,"['climate change', 'culture', 'global issues',...",0.512684
1,a global pandemic calls for global solutions,"['coronavirus', 'pandemic', 'epidemiology', 'v...",0.470939
2,lessons from the 1918 flu,"['vaccines', 'business', 'ebola', 'health', 'h...",0.301260
3,how we'll stop polio for good,"['vaccines', 'disease', 'global issues', 'heal...",0.278910
4,a comic sendup of ted2006,"['comedy', 'culture', 'performance', 'humor']",0.243148


In [13]:
# Test with an invalid TED Talk title
invalid_recommendations = recommend_talks("This is an invalid TED Talk title")

# Display the result
invalid_recommendations

TED Talk title not found. Please enter a valid title.


""


In [14]:
# Generate only 3 recommendations
custom_recommendations = recommend_talks(sample_title, top_n=3)

# Display the results
custom_recommendations

,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues'...",0.368081
1,how we can turn the tide on climate,"['climate change', 'environment', 'activism', ...",0.354537
2,a climate solution where all sides can win,"['big problems', 'business', 'china', 'economi...",0.339493


In [15]:
# Display recommendations in a readable DataFrame
recommendations.style.format({
    "similarity_score": "{:.4f}"
})

,title,topics,similarity_score
0,new thinking on the climate crisis,"['activism', 'climate change', 'global issues', 'science']",0.3681
1,how we can turn the tide on climate,"['climate change', 'environment', 'activism', 'sustainability', 'social change', 'leadership', 'society', 'science']",0.3545
2,a climate solution where all sides can win,"['big problems', 'business', 'china', 'economics', 'climate change', 'global issues', 'goal-setting', 'policy', 'politics', 'government', 'pollution', 'society']",0.3395
3,a one-man world summit,"['comedy', 'global issues', 'government', 'performance', 'politics', 'humor']",0.3389
4,how we could change the planet's climate future,"['climate change', 'sustainability', 'environment', 'anthropocene', 'future', 'science', 'social change', 'society']",0.3384


## Observations

1. TF-IDF successfully converted the textual content of 4005 TED Talks into 5000 numerical features.
2. Cosine similarity helped identify TED Talks with similar textual content.
3. The recommendation function returned the top 5 similar TED Talks for a selected title.
4. The selected TED Talk was excluded from its own recommendations.
5. Invalid TED Talk titles were handled appropriately.
6. The function also allowed users to customize the number of recommendations using top_n.
7. The recommendation results were displayed in a readable DataFrame with similarity scores.